In [ ]:
from dataclasses import replace, asdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathtracer import Renderer, lattice_preview, display_encode

from stats import pixel_distribution, principal_components
from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq

---
# DATA
Renders the Cornell box once and tabulates each AOV's shape, dtype and range next to the camera that produced it.

- `beauty` and `luminance` are scene-referred linear values in cd/m^2, not display codes
- `rendercam` is the single source of camera state: later sections read their lattice geometry or exposure from it
- `show_sky` makes primary misses return environment radiance, so every pixel sees light rather than a black void
- Change the camera, sky or AOVs here and rerun all; only WHITENING renders again, its second half under the same settings

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
show_sky = True
frame = scene.render(camera=rendercam, aovs=("beauty", "luminance"), show_sky=show_sky)

In [ ]:
stats = {k: dict(shape=v.shape, dtype=v.dtype, min=v.min(), max=v.max(), mean=v.mean()) for k, v in frame.items()}
display(pd.DataFrame(stats).T, pd.Series(asdict(rendercam), name="camera").to_frame())

---
# IMAGE
Beauty as the viewer displays it: lattice preview, 8-bit code statistics and each RGB channel in grayscale.

- `lattice_preview` draws each pixel as its true cell (hexagon or square), so the hex row offset is visible
- Stats and channels use `display_encode` output: camera exposure, OCIO display transform, dither, then 8-bit quantise
- Statistics therefore describe the displayed image, not scene radiance; dither fills codes a plain quantise would leave empty

In [ ]:
x = frame["beauty"]

plt.imshow(display_encode(lattice_preview(x, rendercam, scale=8), camera=rendercam))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# 8-bit stats of beauty as the viewer displays it: pixel_distribution bins codes 0-255, so it takes display_encode output, not cd/m^2
encoded = display_encode(x, camera=rendercam)

pixel_count, stat = pixel_distribution(encoded)

labels = ("shape", "pixels", "min", "max", "least_frequent", "most_frequent", "mean", "std", "median", "nonzero", "empty_bins", "unique_colours")
display(pd.Series((encoded.shape, *stat), index=labels, name="beauty").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=[18, 5])
for c, name in enumerate(("R", "G", "B")):
    ax[c].imshow(encoded[..., c], cmap="gray")
    ax[c].set_title(name)
    ax[c].set_xticks([]), ax[c].set_yticks([])
plt.tight_layout()

---
# SPECTRAL
Luminance spectrum on the camera's pixel lattice: where energy sits by frequency and direction, and how far above the noise floor.

- Lattice-aware DFT: a per-row twiddle moves odd rows to their true positions, so square and hex share one path
- Frequencies are cycles per column pitch; each bin is folded to its nearest reciprocal-lattice point for its true radius and angle
- Radial power excludes DC and stops at the lattice Nyquist; the spatial frequency curve is f·P(f), so its peak is the dominant scale
- Phase is referenced to the luminance centroid, so it does not depend on where the scene sits in frame
- Log magnitude is dB re the peak bin; zero bins are NaN and left out of the noise floor histogram

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
img = frame["luminance"][..., 0]
twiddle = row_twiddle(img.shape, row_shift)
spectrum = fft(img, twiddle)
fy, fx = bin_freq(img.shape, row_pitch)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = np.sqrt(radius_sq)
mag = np.abs(spectrum)

bin_width = 1.0 / min(img.shape[1], img.shape[0] * row_pitch)
nyquist = 0.5 * min(1.0 / row_pitch, np.hypot(1.0, row_shift / row_pitch))
# DC excluded
power, edges = np.histogram(radius, bins=np.arange(0.0, radius.max() + bin_width, bin_width), weights=np.where(radius > 0, mag**2, 0.0))
centres = 0.5 * (edges[1:] + edges[:-1])
inside = edges[1:] <= nyquist
log_power = centres * power
peak = np.argmax(np.where(inside, log_power, 0.0))
print(f"Frequency at which power per log frequency is greatest: {centres[peak]:.4f} cycles/pitch")

total = img.sum()
cx = (np.arange(img.shape[1]) @ img.sum(axis=0) + row_shift * img[1::2].sum()) / total
cy = row_pitch * (np.arange(img.shape[0]) @ img.sum(axis=1)) / total
phase = np.angle(spectrum * np.exp(2j * np.pi * (fx[None, :] * cx + fy[:, None] * cy)))
log_mag = 20 * np.log10(mag / mag.max(), out=np.full_like(mag, np.nan), where=mag > 0)

In [ ]:
fig, (ax_lum, ax_mag, ax_phase) = plt.subplots(1, 3, figsize=[30,10])
ax_lum.imshow(img)
ax_lum.set_title("Luminance")

ax_mag.imshow(np.fft.fftshift(log_mag), cmap="binary")
ax_mag.set_title("Log magnitude")

ax_phase.imshow(np.fft.fftshift(phase), cmap="binary")
ax_phase.set_title("Phase")

In [ ]:
power_2d = mag**2
theta = np.arctan2(true_fy, true_fx)

fig, ax = plt.subplots(1, 5, figsize=[25, 5])

ax[0].plot(centres[inside], log_power[inside], lw=1)
ax[0].axvline(centres[peak], color="r", lw=1)
ax[0].set_title("Spatial frequency")
ax[0].set_xlabel("Radius (cycles/pitch)"), ax[0].set_ylabel("Power per log frequency")
ax[0].set_ylim(bottom=0)

ax[1].plot(edges[1:], np.cumsum(power) / power.sum(), lw=1)
ax[1].set_title("Cumulative energy")
ax[1].set_xlabel("Radius (cycles/pitch)"), ax[1].set_ylabel("Fraction of AC power")

ax[2].semilogy(np.fft.fftshift(fx), np.fft.fftshift(power_2d.sum(axis=0)), lw=1, label="fx")
ax[2].semilogy(np.fft.fftshift(fy), np.fft.fftshift(power_2d.sum(axis=1)), lw=1, label="fy")
ax[2].set_title("Marginal spectra")
ax[2].set_xlabel("Frequency (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].legend()

ax[3].hist(log_mag[np.isfinite(log_mag)], bins=100)
ax[3].set_title("Noise floor")
ax[3].set_xlabel("Magnitude (dB re peak)"), ax[3].set_ylabel("Bins")

ring = (radius > 0) & (radius <= nyquist)
ax[4].hist(theta[ring], bins=36, range=(-np.pi, np.pi), weights=power_2d[ring])
ax[4].set_title("Angular power")
ax[4].set_xlabel("Angle (rad)"), ax[4].set_ylabel("Power")

for a in ax: a.grid(alpha=0.25)

---
# WHITENING
Log luminance through the Atick & Redlich retinal filter: pairwise correlations removed where signal beats noise, edges left behind.

- Input is `log(L + dark)`, the photoreceptor's Weber compression; dark light = RMS render noise of the mean (Barlow), finite at L = 0
- Filter `K = sqrt(R) / (R + N)` (Dayan & Abbott eq. 4.49): Wiener gain `R / (R + N)` times whitening `R^-1/2`
- `R` and `N` are measured, not modelled: a second render with an independent sampler scramble (`seed=2`) shares the signal only
- Isotropic like a ganglion cell's centre-surround, so per ring of radius: cross power of the two halves estimates signal `R`, a quarter of their difference power the mean's noise `N`
- Whitened power is `R / (R + N)`: flat (decorrelated) where SNR is high, falling to zero where render noise dominates
- Residual neighbour correlation is the noise roll-off plus the scene's horizontal/vertical bias, which an isotropic filter cannot flatten
- DC is removed, so the output is zero-mean contrast; what survives is edges, the phase structure second-order statistics cannot explain

In [ ]:
# seed 2 rescrambles the sampler, so this half's noise is independent of frame's and the two share signal only
half = scene.render(camera=rendercam, aovs=("luminance",), seed=2, show_sky=show_sky)["luminance"][..., 0]
# Dark light = RMS noise of the mean: Weber-like log above the noise floor, finite at L = 0
dark = np.sqrt(np.mean(0.25 * (img - half) ** 2))
halves = [np.log(x + dark) for x in (img, half)]
first, other = (fft(x, twiddle) for x in halves)

ac = radius > 0
# Clamped so the top edge closes the last ring, as in np.histogram
ring_index = np.minimum(np.digitize(radius, edges) - 1, centres.size - 1)
count = np.bincount(ring_index[ac], minlength=centres.size)
ring_mean = lambda v: np.divide(np.bincount(ring_index[ac], weights=v[ac], minlength=centres.size), count, out=np.full(centres.size, np.nan), where=count > 0)

signal = np.maximum(ring_mean(np.real(first * other.conj())), 0.0)
noise = ring_mean(np.abs(first - other) ** 2 / 4)
gain = np.sqrt(signal) / (signal + noise)
log_spectrum = 0.5 * (first + other)
whitened_spectrum = log_spectrum * np.where(ac, gain[ring_index], 0.0)
whitened = ifft(whitened_spectrum, twiddle).real
whitened_power = ring_mean(np.abs(whitened_spectrum) ** 2)

neighbour = lambda x: np.corrcoef(x[:, :-1].ravel(), x[:, 1:].ravel())[0, 1]
print(f"Horizontal neighbour correlation: input {neighbour(0.5 * (halves[0] + halves[1])):.3f}, whitened {neighbour(whitened):.3f}")

In [ ]:
preview = lattice_preview(whitened[..., None], rendercam)
# Image column weighted by its aspect and plots square, so all three panels share one height on either lattice
fig, ax = plt.subplots(1, 3, figsize=[24, 5], width_ratios=[preview.shape[1] / preview.shape[0], 1, 1])

limit = np.abs(whitened).max()
ax[0].imshow(preview, cmap="gray", vmin=-limit, vmax=limit)
ax[0].set_title("Whitened")
ax[0].set_xticks([]), ax[0].set_yticks([])

ax[1].semilogy(centres[inside], signal[inside], lw=1, label="Signal R")
ax[1].semilogy(centres[inside], noise[inside], lw=1, label="Noise N")
ax[1].set_title("Signal and noise")
ax[1].set_xlabel("Radius (cycles/pitch)"), ax[1].set_ylabel("Power")
ax[1].legend()

ax[2].plot(centres[inside], whitened_power[inside], lw=1)
ax[2].set_title("Whitened power")
ax[2].set_xlabel("Radius (cycles/pitch)"), ax[2].set_ylabel("R / (R + N)")
ax[2].set_ylim(bottom=0)

for a in ax[1:]: a.set_box_aspect(1), a.grid(alpha=0.25)

---
# PHASE
Each image keeps its power spectrum but takes random phase: what survives the scramble is the power spectrum's share of the structure.

- Oppenheim & Lim 1981: edges need the phases of many frequencies aligned at one place; magnitude alone fixes only pairwise correlation
- Each pair shares one power spectrum exactly, so pairwise pixel correlations match; the scrambled twin keeps none of the edges
- Random phase is the angle of the lattice DFT of white Gaussian noise: uniform, and conjugate-symmetric, so the image stays real
- DC is removed from every image, so all four are zero-mean and compare like for like

In [ ]:
rng = np.random.default_rng(0)
random_phase = np.exp(1j * np.angle(fft(rng.standard_normal(img.shape), twiddle)))
log_ac = np.where(ac, log_spectrum, 0.0)

images = {
    "Log luminance": ifft(log_ac, twiddle).real,
    "Random phase": ifft(np.abs(log_ac) * random_phase, twiddle).real,
    "Whitened": whitened,
    "Whitened, random phase": ifft(np.abs(whitened_spectrum) * random_phase, twiddle).real,
}

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=[20, 10])
for a, (name, x) in zip(ax.ravel(), images.items()):
    limit = np.abs(x).max()
    a.imshow(lattice_preview(x[..., None], rendercam), cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

---
# SPARSITY
Value distributions of the PHASE images: whitened images are sparse (heavy-tailed), their random-phase twins Gaussian.

- Excess kurtosis `m4 / m2^2 - 3` is 0 for a Gaussian and 3 for a Laplace; positive means super-Gaussian, i.e. sparse (Field 1994)
- A random-phase twin sums many waves of independent phase, so the central limit theorem pulls it towards Gaussian
- That pull is weak when power sits in a few low frequencies, so the log luminance twin can stay measurably off zero
- Whitening spreads power over all frequencies: its twin is Gaussian, while the true-phase image keeps the heavy tails of its edges
- Histograms are in standard deviations (every image is zero-mean); the references are the unit-variance Gaussian and Laplace densities

In [ ]:
kurtosis = lambda x: np.mean(x ** 4) / np.var(x) ** 2 - 3
display(pd.Series({name: kurtosis(x) for name, x in images.items()}, name="excess kurtosis").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[16, 8])
for a, pair in zip(ax, (("Log luminance", "Random phase"), ("Whitened", "Whitened, random phase"))):
    standard = [images[name] / images[name].std() for name in pair]
    density = [a.hist(x.ravel(), bins="auto", density=True, histtype="step", log=True, label=name)[0] for name, x in zip(pair, standard)]
    reach = max(np.abs(x).max() for x in standard)
    z = np.linspace(-reach, reach)
    a.plot(z, np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi), "k--", lw=1, label="Gaussian")
    a.plot(z, np.exp(-np.sqrt(2) * np.abs(z)) / np.sqrt(2), "k:", lw=1, label="Laplace")
    # Floor at one sample in a bin, the lowest density the histograms can show, so the reference tails don't squash the data
    a.set_ylim(bottom=min(d[d > 0].min() for d in density))
    a.set_title(pair[0])
    a.set_xlabel("Standard deviations"), a.set_ylabel("Density")
    a.legend()
    a.set_box_aspect(1), a.grid(alpha=0.25)

---
# PCA
Principal axes of beauty's RGB values: how much colour variance each axis carries and where in the image it lives.

- Runs on linear beauty, not the encoded image, so axes reflect radiance rather than display gamma
- PC1 close to (1, 1, 1)/√3 means variation is mostly brightness; PC2 and PC3 then carry chroma
- Projection is each pixel's coordinate along that axis; distance is its distance from that axis (norm over the other two)

In [ ]:
_, variance, axes, projection = principal_components(frame["beauty"])
display(pd.DataFrame(axes.T, index=("PC1", "PC2", "PC3"), columns=("R", "G", "B")).assign(variance=variance, explained=variance / variance.sum()))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=[18, 7])
for i in range(3):
    ax[0, i].imshow(projection[..., i])
    ax[0, i].set_title(f"PC{i + 1} projection")
    distance = np.linalg.norm(np.delete(projection, i, axis=-1), axis=-1)
    ax[1, i].imshow(distance, cmap="gray")
    ax[1, i].set_title(f"PC{i + 1} distance")
for a in ax.flat: a.set_xticks([]), a.set_yticks([])
plt.tight_layout()